# Feature engineering regresijoje (Python, `tips` iš seaborn)

Šiame faile parodoma, kaip iš žalių duomenų sukurti prasmingus požymius (features), kurie padeda **pagerinti regresijos modelio tikslumą** ir **palengvina interpretaciją**.

Naudojamas `tips` duomenų rinkinys (restorano sąskaitos ir arbatpinigiai). Tikslinis kintamasis: **`tip`** (arbatpinigiai).


## 1. Bibliotekos ir duomenys

Duomenys paimami iš `seaborn` bibliotekos.


In [1]:
import numpy as np
import pandas as pd
import seaborn as sns

import statsmodels.api as sm
from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error


In [2]:
# Duomenys
tips = sns.load_dataset("tips").copy()

tips.tail()

,total_bill,tip,sex,smoker,day,time,size
239,29.03,5.92,Male,No,Sat,Dinner,3
240,27.18,2.00,Female,Yes,Sat,Dinner,2
241,22.67,2.00,Male,Yes,Sat,Dinner,2
242,17.82,1.75,Male,No,Sat,Dinner,2
243,18.78,3.00,Female,No,Thur,Dinner,2


## 2. Bazinis modelis (be feature engineering)

Paprastas atskaitos taškas: arbatpinigių (`tip`) prognozė pagal sąskaitos sumą (`total_bill`).


In [3]:
# Bazinis modelis: tip ~ total_bill
X_base = sm.add_constant(tips[["total_bill"]])
y = tips["tip"]

m_base = sm.OLS(y, X_base).fit()
m_base.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                    tip   R-squared:                       0.457
Model:                            OLS   Adj. R-squared:                  0.454
Method:                 Least Squares   F-statistic:                     203.4
Date:                Fri, 27 Feb 2026   Prob (F-statistic):           6.69e-34
Time:                        12:50:03   Log-Likelihood:                -350.54
No. Observations:                 244   AIC:                             705.1
Df Residuals:                     242   BIC:                             712.1
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.9203      0.160      5.761      0.000       0.606       1.235
total_bill     0.1050      0.007     14.260      0.000       0.091       0.120
==============================================================================
Omnibus:                       20.185   Durbin-Watson:                   2.151
Prob(Omnibus):                  0.000   Jarque-Bera (JB):               37.750
Skew:                           0.443   Prob(JB):                     6.35e-09
Kurtosis:                       4.711   Cond. No.                         53.0
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

### Interpretacija (bazinis modelis)

- `total_bill` koeficientas rodo, kiek **vidutiniškai** pasikeičia `tip`, kai `total_bill` padidėja 1 vienetu, laikant kitus kintamuosius pastoviais.
- Tai paprastas tiesinis ryšys. Jei EDA rodo „lenktą“ priklausomybę, gali padėti polinominiai nariai arba binning.


## Bazinio regresijos modelio interpretacija

**Modelio paskirtis:**  
Modelis prognozuoja arbatpinigius (`tip`) pagal sąskaitos sumą (`total_bill`).

### Pagrindiniai rodikliai
- **R² = 0.457**  
  Modelis paaiškina apie **45.7 %** arbatpinigių variacijos. Tai vidutinio stiprumo modelis, tinkamas kaip atskaitos taškas.
- **Adj. R² = 0.454**  
  Patvirtina, kad modelio paaiškinamoji galia nėra dirbtinai išpūsta.
- **F-statistic p-value < 0.001**  
  Modelis statistiškai reikšmingas kaip visuma.

### Koeficientų interpretacija
- **Intercept (0.92)**  
  Teorinė prognozuojama arbatpinigių reikšmė, kai sąskaitos suma lygi 0. Praktinės verslo prasmės beveik neturi, bet reikalinga modelio matematikai.
- **total_bill (0.105)**  
  Kai sąskaitos suma padidėja **1 piniginiu vienetu**, arbatpinigiai vidutiniškai padidėja apie **0.105**, laikant kitus veiksnius pastoviais.  
  Ryšys yra **teigiamas ir statistiškai reikšmingas** (p < 0.001).

### Verslo išvada
- Sąskaitos suma yra **svarbus arbatpinigių veiksnys**, bet paaiškina tik dalį elgsenos.
- Tikėtina, kad arbatpinigiams įtaką daro ir kiti veiksniai (staliuko dydis, savaitės diena, rūkymas ir pan.).
- Šis modelis tinka kaip **bazinis palyginimo modelis**, prieš taikant feature engineering.


## 3. Cross-validation (CV) baziniam modeliui

CV padeda įvertinti, ar patobulinimai duoda naudą **stabiliai**, o ne tik viename duomenų padalinime.


In [4]:
# CV su scikit-learn (R2)
X_cv_base = tips[["total_bill"]].values
y_cv = y.values

cv = KFold(n_splits=5, shuffle=True, random_state=42)
lr = LinearRegression()

r2_scores_base = cross_val_score(lr, X_cv_base, y_cv, cv=cv, scoring="r2")
r2_scores_base, r2_scores_base.mean(), r2_scores_base.std()

(array([0.54493817, 0.20104191, 0.55967082, 0.47616396, 0.33495202]),
 0.42335337410555907,
 0.13666854246609578)

## Cross-validation interpretacija (bazinis modelis)

**Naudota metrika:** R²  
**CV padalijimų skaičius:** 5

### Rezultatai
- **R² reikšmės per dalis:**  
  0.545, 0.201, 0.560, 0.476, 0.335
- **Vidutinis R²:** **0.423**
- **Standartinis nuokrypis:** **0.137**

### Interpretacija
- Vidutinis R² rodo, kad bazinis modelis **paaiškina apie 42 % variacijos** naujuose duomenyse.
- R² reikšmės tarp padalijimų **stipriai svyruoja**, kas rodo, kad modelio stabilumas yra ribotas.
- Kai kuriuose padalijimuose modelis veikia pakankamai gerai, kituose – gerokai prasčiau.

### Verslo išvada
- Modelis fiksuoja bendrą ryšį tarp sąskaitos sumos ir arbatpinigių, bet **nepakankamai stabiliai**.
- Tai signalas, kad **trūksta svarbių požymių**, o feature engineering gali reikšmingai pagerinti rezultatą.

### Praktinė išvada
- Bazinis modelis tinka kaip **atskaitos taškas**.
- Visi patobulinimai turi būti vertinami pagal tai, ar jie **didina vidutinį CV R² ir mažina jo svyravimus**.


## 4. Feature engineering technikos

Dažniausiai praktikoje naudojama:

- Skaitiniams kintamiesiems: **polinominiai nariai**, **kintamųjų kombinacijos**, **interaction terms**, **binning**
- Kategoriniams kintamiesiems: **binary columns**, **dummy variables**, **kategorijų grupavimas (binning)**


## 5. Polinominiai nariai (polynomial terms)

Polinominiai nariai (`x²`, `x³`) padeda, kai ryšys tarp požymio ir tikslo yra **netiesinis**.
Didinant laipsnį dažniausiai didėja overfitting rizika, todėl sudėtingumas parenkamas pagal **CV**.


In [5]:
# Polinominiai nariai: total_bill, total_bill^2, total_bill^3
tips["bill2"] = tips["total_bill"] ** 2
tips["bill3"] = tips["total_bill"] ** 3

X_poly = sm.add_constant(tips[["total_bill", "bill2", "bill3"]])
m_poly = sm.OLS(y, X_poly).fit()
m_poly.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                    tip   R-squared:                       0.473
Model:                            OLS   Adj. R-squared:                  0.466
Method:                 Least Squares   F-statistic:                     71.77
Date:                Fri, 27 Feb 2026   Prob (F-statistic):           3.65e-33
Time:                        12:50:03   Log-Likelihood:                -346.83
No. Observations:                 244   AIC:                             701.7
Df Residuals:                     240   BIC:                             715.7
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.7059      0.680     -1.038      0.300      -2.045       0.633
total_bill     0.3491      0.094      3.722      0.000       0.164       0.534
bill2         -0.0105      0.004     -2.702      0.007      -0.018      -0.003
bill3          0.0001   4.88e-05      2.719      0.007    3.66e-05       0.000
==============================================================================
Omnibus:                       20.263   Durbin-Watson:                   2.182
Prob(Omnibus):                  0.000   Jarque-Bera (JB):               34.235
Skew:                           0.483   Prob(JB):                     3.68e-08
Kurtosis:                       4.560   Cond. No.                     2.59e+05
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 2.59e+05. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

## Polinominio regresijos modelio interpretacija

**Modelio paskirtis:**  
Arbatpinigių (`tip`) prognozė pagal sąskaitos sumą, leidžiant **netiesinį ryšį** tarp `total_bill` ir `tip`.

Modelio forma:
tip = β₀ + β₁·total_bill + β₂·total_bill² + β₃·total_bill³

### Pagrindiniai rodikliai
- **R² = 0.473**  
  Modelis paaiškina apie **47.3 %** arbatpinigių variacijos.
- **Adj. R² = 0.466**  
  Paaiškinamoji galia padidėjo, atsižvelgiant į papildomus kintamuosius.
- **F-statistic p-value < 0.001**  
  Modelis statistiškai reikšmingas kaip visuma.

### Koeficientų interpretacija
- **total_bill (β₁ = 0.349)**  
  Pagrindinis teigiamas sąskaitos poveikis arbatpinigiams.
- **total_bill² (β₂ = −0.0105)**  
  Neigiamas kvadratinis terminas rodo, kad augant sąskaitai, arbatpinigių augimo tempas **lėtėja**.
- **total_bill³ (β₃ ≈ 0.0001)**  
  Trečio laipsnio terminas leidžia modeliui fiksuoti **sudėtingesnę kreivę**, ypač didesnių sąskaitų srityje.
- Visi polinominiai nariai yra **statistiškai reikšmingi** (p < 0.01).

### Verslo interpretacija
- Arbatpinigių ir sąskaitos ryšys nėra tiesinis.
- Didėjant sąskaitai, arbatpinigių augimas **nėra proporcingas** – tam tikrose ribose jis lėtėja ar keičia tempą.
- Modelis geriau atspindi realų elgesį nei paprastas tiesinis modelis.

### Praktinė išvada
- Polinominiai nariai pagerina modelio pritaikymą duomenims.
- Tokį modelį verta rinktis, kai tikslas – **prognozė**, o ne paprasta koeficientų interpretacija.
- Galutinį sprendimą dėl polinomo laipsnio būtina pagrįsti **cross-validation** rezultatais.


In [6]:
# CV polinominiam modeliui
X_cv_poly = tips[["total_bill", "bill2", "bill3"]].values
r2_scores_poly = cross_val_score(lr, X_cv_poly, y_cv, cv=cv, scoring="r2")
r2_scores_poly.mean(), r2_scores_poly.std()

(0.4185958213008357, 0.1358130827404934)

### Dažnos klaidos ir gerosios praktikos (polinomai)

- Klaida: kelti laipsnį „iki kol gražu“ – be CV dažnai gaunamas overfitting.
- Praktika: laikyti kelis variantus (pvz., 2 ir 3 laipsnio) ir rinktis pagal CV.
- Praktika: prieš polinomus įsitikinti, kad nėra akivaizdžių outlier’ių, kurie gali stipriai iškreipti kreivę.


## 6. Kintamųjų kombinacijos (combining features)

Keli stulpeliai sujungiami aritmetiškai: suma, skirtumas, sandauga, santykis.
Tikslas – sukurti **versliškai prasmingą** rodiklį ir kartais sumažinti multikolinearumą.

Pavyzdys: `tip_per_person = tip / size` parodo arbatpinigius vienam asmeniui.


In [7]:
tips["tip_per_person"] = tips["tip"] / tips["size"]
tips[["tip", "size", "tip_per_person"]].head()

,tip,size,tip_per_person
0,1.01,2,0.505000
1,1.66,3,0.553333
2,3.50,3,1.166667
3,3.31,2,1.655000
4,3.61,4,0.902500


## Kintamųjų kombinacijos (combining features)

**Kintamųjų kombinacijos** – tai nauji požymiai, sukurti **aritmetiškai sujungiant esamus kintamuosius**.
Tikslas – geriau atspindėti verslo logiką nei naudojant kintamuosius atskirai.

### Pavyzdys
- `tip` – arbatpinigių suma
- `size` – žmonių skaičius prie stalo
- **`tip_per_person = tip / size`** – arbatpinigiai vienam asmeniui

### Verslo prasmė
- `tip` vienas pats neatsižvelgia į stalo dydį.
- `tip_per_person` leidžia įvertinti **kliento dosnumą**, nepriklausomai nuo žmonių skaičiaus.
- Tai dažnai yra stabilesnis ir labiau palyginamas rodiklis.

### Svarbi pastaba (kritinė klaida)
- `tip_per_person` **negali būti naudojamas kaip požymis**, jei `tip` yra tikslinis kintamasis.
- Tai būtų **target leakage**, nes požymis tiesiogiai naudoja prognozuojamą reikšmę.

### Teisingas analogas
- Vietoje to galima naudoti:
  - `bill_per_person = total_bill / size`
- Tokia kombinacija naudoja tik **įėjimo kintamuosius** ir yra saugi modeliavimui.

### Kada kombinacijos ypač naudingos
- Kai keli kintamieji matuoja tą patį reiškinį iš skirtingų pusių.
- Kai norima sumažinti multikolinearumą.
- Kai reikalinga aiški ir lengvai interpretuojama verslo metrika.

**Esmė:** gerai parinkta kintamųjų kombinacija dažnai perteikia verslo realybę geriau nei atskiri požymiai.


In [8]:
tips["bill_per_person"] = tips["total_bill"] / tips["size"]

X_comb = sm.add_constant(tips[["total_bill", "size", "bill_per_person"]])
m_comb = sm.OLS(y, X_comb).fit()
m_comb.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                    tip   R-squared:                       0.471
Model:                            OLS   Adj. R-squared:                  0.464
Method:                 Least Squares   F-statistic:                     71.13
Date:                Fri, 27 Feb 2026   Prob (F-statistic):           6.02e-33
Time:                        12:50:03   Log-Likelihood:                -347.34
No. Observations:                 244   AIC:                             702.7
Df Residuals:                     240   BIC:                             716.7
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
===================================================================================
                      coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------
const               1.2806      0.577      2.221      0.027       0.145       2.416
total_bill          0.1253      0.030      4.130      0.000       0.066       0.185
size               -0.0356      0.220     -0.162      0.872      -0.469       0.397
bill_per_person    -0.0850      0.075     -1.126      0.261      -0.234       0.064
==============================================================================
Omnibus:                       21.648   Durbin-Watson:                   2.097
Prob(Omnibus):                  0.000   Jarque-Bera (JB):               37.002
Skew:                           0.510   Prob(JB):                     9.23e-09
Kurtosis:                       4.612   Cond. No.                         222.
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

## Kintamųjų kombinacijų modelio interpretacija

**Modelio paskirtis:**  
Arbatpinigių (`tip`) prognozė pagal:
- sąskaitos sumą (`total_bill`),
- žmonių skaičių (`size`),
- sąskaitą vienam asmeniui (`bill_per_person`).

### Pagrindiniai rodikliai
- **R² = 0.471**  
  Modelis paaiškina apie **47.1 %** arbatpinigių variacijos.
- **Adj. R² = 0.464**  
  Panaši reikšmė rodo, kad papildomi kintamieji nepridėjo dirbtinės paaiškinamosios galios.
- **F-statistic p-value < 0.001**  
  Modelis statistiškai reikšmingas kaip visuma.

### Koeficientų interpretacija
- **total_bill (0.125, p < 0.001)**  
  Sąskaitos suma išlieka **stipriausias ir statistiškai reikšmingas** veiksnys.  
  Padidėjus sąskaitai 1 vienetu, arbatpinigiai vidutiniškai padidėja apie **0.125**.
- **size (−0.036, p = 0.872)**  
  Žmonių skaičius neturi statistiškai reikšmingo poveikio arbatpinigiams.
- **bill_per_person (−0.085, p = 0.261)**  
  Sąskaita vienam asmeniui taip pat nėra reikšminga, kai modelyje jau yra `total_bill`.

### Esminė įžvalga
- `total_bill` ir `bill_per_person` yra **stipriai susiję tarpusavyje**.
- Įtraukus abu kintamuosius, jų individualūs efektai tampa sunkiai atskiriami.
- Modelis „pasirenka“ `total_bill` kaip pagrindinį signalą.

### Diagnostinės pastabos
- **Durbin–Watson ≈ 2.10**  
  Autokoreliacijos problemų nėra.
- **Condition Number = 222**  
  Vidutinis lygis – galimas koreliacijos signalas, bet kritinių problemų nėra.

### Verslo išvada
- Arbatpinigius geriausiai paaiškina **bendra sąskaitos suma**, o ne jos struktūra vienam asmeniui.
- `bill_per_person` šiuo atveju **neprideda papildomos informacijos**.
- Paprastesnis modelis su mažiau, bet stipresnių požymių yra labiau pagrįstas.

### Praktinė išvada
- Kintamųjų kombinacijos turi būti naudojamos **vietoje**, o ne **kartu su stipriai koreliuojančiais originaliais kintamaisiais**.
- Jei pasirenkamas `bill_per_person`, dažnai verta atsisakyti `total_bill` ir `size`.

**Esmė:** kintamųjų kombinacijos yra naudingos tik tada, kai jos iš tikrųjų atneša naują informaciją, o ne dubliuoja jau esamą signalą.


### Pastaba apie interpretaciją (kombinacijos)

Kai modelyje yra ir originalūs kintamieji, ir jų kombinacijos (pvz., `total_bill`, `size`, `total_bill/size`),
koeficientai gali tapti mažiau intuityvūs dėl koreliacijų. Tokiais atvejais naudinga:
- patikrinti koreliacijas,
- naudoti VIF,
- arba palikti tik prasmingiausią reprezentaciją (pvz., tik `bill_per_person`).


## 7. Interaction terms

Interaction term fiksuoja situaciją, kai vieno kintamojo poveikis tikslui priklauso nuo kito kintamojo.
Tipinis pavyzdys: `total_bill * smoker` – ar sąskaitos poveikis arbatpinigiams skiriasi rūkantiems ir nerūkantiems.


### 7.1 Binary column: `smoker_yes`

Kategorinis kintamasis paverčiamas į 0/1. Tai paprasčiausia forma.


In [9]:
tips["smoker_yes"] = np.where(tips["smoker"] == "Yes", 1, 0)

# Svarbu: konvertavimas į float (ypač naudojant statsmodels) padeda išvengti ValueError dėl tipų
tips["smoker_yes"] = tips["smoker_yes"].astype(float)

tips[["smoker", "smoker_yes"]].head()

,smoker,smoker_yes
0,No,0.0
1,No,0.0
2,No,0.0
3,No,0.0
4,No,0.0


### 7.2 Modelis su interaction term

- `total_bill` koeficientas – bazinės grupės (kai `smoker_yes=0`) nuolydis.
- `smoker_yes` koeficientas – intercept poslinkis tarp grupių (kai `total_bill=0`).
- `total_bill_x_smoker` koeficientas – nuolydžio skirtumas tarp grupių.


In [10]:
tips["total_bill_x_smoker"] = (tips["total_bill"] * tips["smoker_yes"]).astype(float)

X_int = sm.add_constant(tips[["total_bill", "smoker_yes", "total_bill_x_smoker"]].astype(float))
m_int = sm.OLS(y, X_int).fit()
m_int.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                    tip   R-squared:                       0.506
Model:                            OLS   Adj. R-squared:                  0.500
Method:                 Least Squares   F-statistic:                     81.95
Date:                Fri, 27 Feb 2026   Prob (F-statistic):           1.56e-36
Time:                        12:50:03   Log-Likelihood:                -338.91
No. Observations:                 244   AIC:                             685.8
Df Residuals:                     240   BIC:                             699.8
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
=======================================================================================
                          coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------
const                   0.3601      0.202      1.782      0.076      -0.038       0.758
total_bill              0.1372      0.010     14.172      0.000       0.118       0.156
smoker_yes              1.2042      0.312      3.856      0.000       0.589       1.819
total_bill_x_smoker    -0.0676      0.014     -4.762      0.000      -0.096      -0.040
==============================================================================
Omnibus:                       45.973   Durbin-Watson:                   2.078
Prob(Omnibus):                  0.000   Jarque-Bera (JB):              122.757
Skew:                           0.830   Prob(JB):                     2.21e-27
Kurtosis:                       6.053   Cond. No.                         132.
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

## Interaction term modelio interpretacija

**Modelio paskirtis:**  
Arbatpinigių (`tip`) prognozė pagal sąskaitos sumą (`total_bill`), rūkymo faktą (`smoker_yes`)
ir jų sąveiką (`total_bill × smoker_yes`).

Modelio forma:  
tip = β₀ + β₁·total_bill + β₂·smoker_yes + β₃·(total_bill × smoker_yes)

### Pagrindiniai rodikliai
- **R² = 0.506**  
  Modelis paaiškina apie **50.6 %** arbatpinigių variacijos – tai aiškus pagerėjimas lyginant su baziniu modeliu.
- **Adj. R² = 0.500**  
  Papildomi kintamieji sukuria realią, o ne dirbtinę vertę.
- **F-statistic p-value < 0.001**  
  Modelis statistiškai reikšmingas kaip visuma.

### Koeficientų interpretacija
- **total_bill (0.137, p < 0.001)**  
  Nerūkantiems klientams, padidėjus sąskaitai 1 vienetu, arbatpinigiai vidutiniškai padidėja apie **0.137**.
- **smoker_yes (1.204, p < 0.001)**  
  Rūkantys klientai turi **aukštesnį pradinį arbatpinigių lygį** (intercepto poslinkį), kai `total_bill` artimas nuliui.
- **total_bill × smoker_yes (−0.068, p < 0.001)**  
  Rūkantiems klientams sąskaitos augimo poveikis arbatpinigiams yra **silpnesnis** nei nerūkantiems.

### Kaip tai suprasti versliškai
- Nerūkantiems klientams arbatpinigiai **auga sparčiau** didėjant sąskaitai.
- Rūkantiems klientams arbatpinigiai pradžioje yra didesni, bet **didėjant sąskaitai auga lėčiau**.
- Tai reiškia, kad sąskaitos dydžio poveikis **priklauso nuo kliento tipo**.

### Verslo išvada
- Rūkymo faktas **keičia sąskaitos poveikį arbatpinigiams**, todėl interaction term čia yra prasmingas.
- Modelis ne tik tikslesnis, bet ir suteikia **aiškią elgsenos įžvalgą**.

### Praktinė išvada
- Interaction terms verta naudoti tada, kai yra pagrįsta hipotezė, kad vieno veiksnio poveikis priklauso nuo kito.
- Šis modelis yra geras pavyzdys, kai didesnis sudėtingumas yra pateisinamas aiškia papildoma verte.

**Esmė:** interaction term leidžia modeliui atskleisti elgsenos skirtumus, kurių paprastas modelis nemato.


In [11]:
# CV modeliams: bazinis vs interaction
X_cv_int = tips[["total_bill", "smoker_yes", "total_bill_x_smoker"]].values.astype(float)

r2_scores_int = cross_val_score(lr, X_cv_int, y_cv, cv=cv, scoring="r2")
r2_scores_base.mean(), r2_scores_int.mean(), r2_scores_int.std()

(0.42335337410555907, 0.43434506537049417, 0.10427919804381915)

## Cross-validation palyginimas: bazinis vs interaction modelis

**Naudota metrika:** R²  
**CV metodas:** 5-fold cross-validation

### Rezultatai
- **Bazinis modelis (tip ~ total_bill):**  
  Vidutinis R² = **0.423**
- **Interaction modelis (tip ~ total_bill + smoker + total_bill×smoker):**  
  Vidutinis R² = **0.434**
- **Interaction modelio R² standartinis nuokrypis:** **0.104**

### Interpretacija
- Interaction modelis **vidutiniškai veikia geriau** nei bazinis modelis.
- R² padidėjimas yra **nedidelis**, bet nuoseklus.
- Standartinis nuokrypis rodo, kad modelio rezultatai **nėra itin stabilūs**, tačiau neblogesni nei bazinio.

### Verslo išvada
- Rūkymo faktas iš tiesų **modifikuoja sąskaitos poveikį arbatpinigiams**.
- Interaction term sukuria **papildomą prognozinę vertę**, nors pagerėjimas nėra drastiškas.

### Praktinė išvada
- Interaction term šiame atvejyje yra **pateisinamas**, nes:
  - pagerina vidutinį CV R²,
  - suteikia interpretuojamą elgsenos įžvalgą.
- Sprendimas naudoti interaction modelį priklauso nuo to, ar **nedidelis tikslumo padidėjimas** yra vertas papildomo sudėtingumo.

**Esmė:** interaction term sukuria apčiuopiamą, bet ribotą prognozinę naudą, kuri turi būti vertinama kartu su interpretacijos verte.


### Dažnos klaidos ir gerosios praktikos (interaction)

- Klaida: pridėti `x1*x2` ir išmesti originalius `x1` ar `x2` – tada interpretacija tampa neteisinga.
- Praktika: interaction term visada naudojamas **kartu** su originaliais kintamaisiais.
- Praktika: sprendimas turi būti pagrįstas EDA arba CV rezultatais, nes interaction term didina modelio sudėtingumą.


## 8. Kategoriniai kintamieji ir dummy variables

Regresijoje kategorijos turi būti paverstos skaitiniais stulpeliais.
`pd.get_dummies()` sukuria one-hot encoding.
Linear regression atveju paprastai reikia pašalinti vieną kategoriją (`drop_first=True`) dėl tobulos multikolinearumo.


In [12]:
# Dummy variables: day ir time
dummies = pd.get_dummies(tips[["day", "time"]], drop_first=True)

# Svarbu: konvertuoti į float, kad statsmodels visur matytų skaitinius tipus
dummies = dummies.astype(float)

dummies.head()

,day_Fri,day_Sat,day_Sun,time_Dinner
0,0.0,0.0,1.0,1.0
1,0.0,0.0,1.0,1.0
2,0.0,0.0,1.0,1.0
3,0.0,0.0,1.0,1.0
4,0.0,0.0,1.0,1.0


### Modelis su dummy kintamaisiais

Interpretacija:
- Kiekvieno dummy koeficientas rodo skirtumą nuo reference level (bazinės kategorijos), laikant kitus kintamuosius pastoviais.
- Bazinė kategorija yra ta, kuri išmesta `drop_first=True` metu.


In [13]:
X_cat = pd.concat([tips[["total_bill"]], dummies], axis=1).astype(float)
X_cat = sm.add_constant(X_cat)

m_cat = sm.OLS(y, X_cat).fit()
m_cat.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                    tip   R-squared:                       0.459
Model:                            OLS   Adj. R-squared:                  0.448
Method:                 Least Squares   F-statistic:                     40.39
Date:                Fri, 27 Feb 2026   Prob (F-statistic):           5.56e-30
Time:                        12:50:03   Log-Likelihood:                -350.00
No. Observations:                 244   AIC:                             712.0
Df Residuals:                     238   BIC:                             733.0
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
===============================================================================
                  coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------
const           0.9191      0.187      4.923      0.000       0.551       1.287
total_bill      0.1049      0.008     13.844      0.000       0.090       0.120
day_Fri         0.0855      0.384      0.222      0.824      -0.672       0.843
day_Sat         0.0387      0.468      0.083      0.934      -0.883       0.961
day_Sun         0.1991      0.470      0.424      0.672      -0.726       1.124
time_Dinner    -0.1080      0.444     -0.243      0.808      -0.984       0.768
==============================================================================
Omnibus:                       20.810   Durbin-Watson:                   2.151
Prob(Omnibus):                  0.000   Jarque-Bera (JB):               38.723
Skew:                           0.459   Prob(JB):                     3.90e-09
Kurtosis:                       4.722   Cond. No.                         277.
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

## Modelio su dummy kintamaisiais interpretacija

**Modelio paskirtis:**  
Arbatpinigių (`tip`) prognozė pagal sąskaitos sumą (`total_bill`) ir kategorinius veiksnius:
- savaitės dieną (`day`),
- laiką (`time`).

Naudojami dummy kintamieji su **reference level** (bazine kategorija).

### Pagrindiniai rodikliai
- **R² = 0.459**  
  Modelis paaiškina apie **45.9 %** arbatpinigių variacijos.
- **Adj. R² = 0.448**  
  Papildomi kategoriniai kintamieji beveik nepadidina paaiškinamosios galios.
- **F-statistic p-value < 0.001**  
  Modelis statistiškai reikšmingas kaip visuma.

### Koeficientų interpretacija
- **total_bill (0.105, p < 0.001)**  
  Sąskaitos suma išlieka pagrindinis arbatpinigių veiksnys.
- **day_Fri, day_Sat, day_Sun**  
  Šių dienų koeficientai rodo skirtumą nuo bazinės dienos (reference level).  
  Visi jie **statistiškai nereikšmingi** (p > 0.8).
- **time_Dinner (−0.108, p = 0.808)**  
  Vakarienės laikas nesukuria statistiškai reikšmingo skirtumo lyginant su baziniu laiku.

### Esminė įžvalga
- Savaitės diena ir valgymo laikas **nepaaiškina papildomos arbatpinigių variacijos**, kai jau žinoma sąskaitos suma.
- Dummy kintamieji padidina modelio sudėtingumą, bet **nesukuria realios prognozinės vertės**.

### Diagnostinės pastabos
- **Durbin–Watson ≈ 2.15**  
  Autokoreliacijos požymių nėra.
- **Condition Number = 277**  
  Galimas vidutinio lygio koreliacijos signalas, bet kritinių problemų nėra.

### Verslo išvada
- Arbatpinigių dydį daugiausia lemia **kiek klientai išleidžia**, o ne kada jie lankosi.
- Šie kategoriniai kintamieji šiame duomenų rinkinyje nėra svarbūs prognozei.

### Praktinė išvada
- Dummy kintamuosius verta laikyti tik tada, jei:
  - jie pagerina CV rezultatus,
  - arba reikalingi verslo interpretacijai.
- Šiuo atveju paprastesnis modelis yra **ne blogesnis**.

**Esmė:** ne visi kategoriniai kintamieji automatiškai pagerina modelį – jų vertę reikia pagrįsti duomenimis.


## 9. Kategorijų grupavimas (binning categorical data)

Jei kategorijų daug arba kai kurios retos, dummy stulpelių skaičius išauga, o tai didina variance ir overfitting riziką.
Sprendimas – sujungti kategorijas į mažiau, bet prasmingų grupių.

Pavyzdys: `day` sujungiamas į `weekday` ir `weekend`.


In [14]:
tips["day_group"] = np.where(tips["day"].isin(["Sat", "Sun"]), "weekend", "weekday")

tips["day_group"].value_counts()

day_group
weekend    163
weekday     81
Name: count, dtype: int64

In [15]:
d_day = pd.get_dummies(tips[["day_group"]], drop_first=True).astype(float)
X_daybin = pd.concat([tips[["total_bill"]], d_day], axis=1).astype(float)
X_daybin = sm.add_constant(X_daybin)

m_daybin = sm.OLS(y, X_daybin).fit()
m_daybin.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                    tip   R-squared:                       0.457
Model:                            OLS   Adj. R-squared:                  0.452
Method:                 Least Squares   F-statistic:                     101.3
Date:                Fri, 27 Feb 2026   Prob (F-statistic):           1.20e-32
Time:                        12:50:03   Log-Likelihood:                -350.54
No. Observations:                 244   AIC:                             707.1
Df Residuals:                     241   BIC:                             717.6
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
=====================================================================================
                        coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------
const                 0.9192      0.174      5.282      0.000       0.576       1.262
total_bill            0.1050      0.007     14.004      0.000       0.090       0.120
day_group_weekend     0.0023      0.141      0.016      0.987      -0.276       0.281
==============================================================================
Omnibus:                       20.162   Durbin-Watson:                   2.151
Prob(Omnibus):                  0.000   Jarque-Bera (JB):               37.717
Skew:                           0.442   Prob(JB):                     6.45e-09
Kurtosis:                       4.711   Cond. No.                         62.9
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

## Kategorinių duomenų binning modelio interpretacija

**Modelio paskirtis:**  
Arbatpinigių (`tip`) prognozė pagal sąskaitos sumą (`total_bill`) ir
savaitės dienos grupę (`day_group`), kur:
- `weekday` – bazinė (reference) kategorija,
- `weekend` – sujungta kategorija (penktadienis–sekmadienis).

### Pagrindiniai rodikliai
- **R² = 0.457**  
  Modelis paaiškina apie **45.7 %** arbatpinigių variacijos.
- **Adj. R² = 0.452**  
  Paaiškinamoji galia beveik nesikeičia, palyginus su baziniu modeliu.
- **F-statistic p-value < 0.001**  
  Modelis statistiškai reikšmingas kaip visuma.

### Koeficientų interpretacija
- **total_bill (0.105, p < 0.001)**  
  Sąskaitos suma išlieka pagrindinis ir statistiškai reikšmingas veiksnys.
- **day_group_weekend (0.002, p = 0.987)**  
  Savaitgalio efektas yra praktiškai lygus nuliui ir **statistiškai nereikšmingas**.
  Tai reiškia, kad arbatpinigių dydis nesiskiria tarp darbo dienų ir savaitgalio,
  kai kontroliuojama sąskaitos suma.

### Esminė įžvalga
- Kategorijų sujungimas sumažino modelio sudėtingumą.
- Tačiau net ir po binning **naujos informacijos apie arbatpinigius neatsirado**.

### Diagnostinės pastabos
- **Durbin–Watson ≈ 2.15**  
  Autokoreliacijos problemų nėra.
- **Condition Number = 62.9**  
  Modelis skaitmeniškai stabilus, multikolinearumo problemų nėra.

### Verslo išvada
- Arbatpinigių dydį lemia **kiek klientai išleidžia**, o ne tai, ar vizitas vyksta savaitgalį.
- Savaitės dienos detalizavimas ar grupavimas šiuo atveju nesukuria prognozinės vertės.

### Praktinė išvada
- Kategorijų binning yra naudingas modelio supaprastinimui.
- Tačiau jo vertę visada reikia patvirtinti per **CV arba reikšmingumo testus**.

**Esmė:** binning sumažina sudėtingumą, bet ne garantuoja geresnį modelį – vertę turi pagrįsti duomenys.


## 10. Skaitinių reikšmių binning (binning numeric data)

Skaitinis kintamasis paverčiamas kategorijomis (intervalais). Tai dažnai mažiau tikslu nei žali skaičiai,
bet yra labai interpretuojama, kai norima aprašyti netiesinį ryšį „segmentais“.

Pavyzdys: `total_bill` suskirstomas į intervalus.


In [16]:
# Intervalai (pavyzdiniai): pagal kvantilius
bins = tips["total_bill"].quantile([0, .25, .5, .75, 1.0]).values
# Užtikrinama, kad ribos būtų griežtai didėjančios (kartais kvantiliai gali sutapti mažiems duomenims)
bins = np.unique(bins)
bins

array([ 3.07  , 13.3475, 17.795 , 24.1275, 50.81  ])

In [17]:
tips["bill_bin"] = pd.cut(tips["total_bill"], bins=bins, include_lowest=True)

tips[["total_bill", "bill_bin"]].head()

,total_bill,bill_bin
0,16.99,"(13.348, 17.795]"
1,10.34,"(3.069, 13.348]"
2,21.01,"(17.795, 24.127]"
3,23.68,"(17.795, 24.127]"
4,24.59,"(24.127, 50.81]"


## `pd.cut()` paaiškinimas

`pd.cut()` naudojamas **skaitiniam kintamajam suskirstyti į intervalus (bins)** ir paversti jį kategoriniu.

Šiame pavyzdyje:
- `total_bill` reikšmės suskirstomos į iš anksto apibrėžtus intervalus.
- Kiekvienai reikšmei priskiriama kategorija `bill_bin`, pvz. „maža“, „vidutinė“, „didelė“ sąskaita.

`include_lowest=True` reiškia, kad **mažiausia reikšmė įtraukiama į pirmą intervalą**, taip išvengiant prarastų reikšmių.

Rezultatas:
- `total_bill` lieka skaitinis.
- `bill_bin` tampa **kategoriniu kintamuoju**, kuris vėliau naudojamas su dummy variables.

**Esmė:** `pd.cut()` leidžia supaprastinti netiesinį ryšį, paverčiant jį aiškiais, interpretuojamais segmentais.


Po `pd.cut()` gaunamas kategorinis kintamasis, todėl vėl kuriami dummy stulpeliai.


In [18]:
d_bill = pd.get_dummies(tips[["bill_bin"]], drop_first=True).astype(float)

X_billbin = pd.concat([d_bill], axis=1).astype(float)
X_billbin = sm.add_constant(X_billbin)

m_billbin = sm.OLS(y, X_billbin).fit()
m_billbin.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                    tip   R-squared:                       0.380
Model:                            OLS   Adj. R-squared:                  0.372
Method:                 Least Squares   F-statistic:                     49.07
Date:                Fri, 27 Feb 2026   Prob (F-statistic):           9.05e-25
Time:                        12:50:03   Log-Likelihood:                -366.59
No. Observations:                 244   AIC:                             741.2
Df Residuals:                     240   BIC:                             755.2
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
=============================================================================================
                                coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------------
const                         1.9207      0.140     13.686      0.000       1.644       2.197
bill_bin_(13.348, 17.795]     0.6890      0.198      3.472      0.001       0.298       1.080
bill_bin_(17.795, 24.127]     1.3046      0.198      6.573      0.000       0.914       1.696
bill_bin_(24.127, 50.81]      2.3169      0.198     11.674      0.000       1.926       2.708
==============================================================================
Omnibus:                       68.805   Durbin-Watson:                   1.991
Prob(Omnibus):                  0.000   Jarque-Bera (JB):              267.752
Skew:                           1.104   Prob(JB):                     7.22e-59
Kurtosis:                       7.633   Cond. No.                         4.79
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

## Skaitinių duomenų binning modelio interpretacija

**Modelio paskirtis:**  
Arbatpinigių (`tip`) prognozė pagal **sąskaitos sumos intervalus** (`bill_bin`), gautus naudojant `pd.cut()`.

### Pagrindiniai rodikliai
- **R² = 0.380**  
  Modelis paaiškina apie **38 %** arbatpinigių variacijos.
- **Adj. R² = 0.372**  
  Paaiškinamoji galia yra mažesnė nei modelių su žaliais skaitiniais kintamaisiais.
- **F-statistic p-value < 0.001**  
  Modelis statistiškai reikšmingas kaip visuma.

### Koeficientų interpretacija
- Bazinė kategorija – **mažiausios sąskaitos** (pirmasis intervalas, kuris neparodytas atskiru koeficientu).
- Kiekvienas `bill_bin` koeficientas rodo, **kiek vidutiniškai didesni arbatpinigiai**, lyginant su bazine sąskaitų grupe.

Interpretacija pagal intervalus:
- **(13.35 – 17.80): +0.69**  
  Vidutiniškai didesni arbatpinigiai nei mažiausių sąskaitų atveju.
- **(17.80 – 24.13): +1.30**  
  Dar didesnis arbatpinigių lygis.
- **(24.13 – 50.81): +2.32**  
  Didžiausias skirtumas nuo bazinės grupės.

Visi intervalų koeficientai yra **statistiškai reikšmingi** (p < 0.01).

### Esminė įžvalga
- Didėjant sąskaitos segmentui, arbatpinigiai **didėja laiptais**, o ne tolygiai.
- Modelis aiškiai parodo **segmentinę struktūrą**, bet praranda dalį tikslumo.

### Verslo išvada
- Binning leidžia lengvai komunikuoti skirtumus tarp **mažų, vidutinių ir didelių sąskaitų**.
- Tai patogu aiškinimui, bet silpniau prognozuoja nei modeliai su žaliais skaitiniais kintamaisiais.

### Praktinė išvada
- Skaitinių duomenų binning tinka, kai svarbiausia **interpretacija ir stabilumas**.
- Jei tikslas – **maksimalus prognozės tikslumas**, geriau naudoti žalius skaitinius kintamuosius arba polinomus.

**Esmė:** binning paverčia tolydų ryšį į aiškius segmentus, aukojant dalį modelio tikslumo.


### Palyginimas su CV (raw vs bins)

Čia tikslas – palyginti, ar binning bent iš dalies pagauna netiesiškumą, palyginus su paprastu tiesiniu modeliu.


In [19]:
# CV: raw total_bill vs bill bins
X_cv_billbin = d_bill.values.astype(float)

r2_scores_billbin = cross_val_score(lr, X_cv_billbin, y_cv, cv=cv, scoring="r2")

r2_scores_base.mean(), r2_scores_billbin.mean(), r2_scores_billbin.std()

(0.42335337410555907, 0.3453992812628664, 0.09698458754323665)

## Cross-validation palyginimas: raw reikšmės vs binning

**Naudota metrika:** R²  
**CV metodas:** 5-fold cross-validation

### Rezultatai
- **Modelis su žaliomis reikšmėmis (`total_bill`):**  
  Vidutinis R² = **0.423**
- **Modelis su binning (`bill_bin`):**  
  Vidutinis R² = **0.345**
- **Binning modelio R² standartinis nuokrypis:** **0.097**

### Interpretacija
- Modelis su žaliu skaitiniu kintamuoju **prognozuoja geriau** nei modelis su binning.
- Binning **praranda dalį informacijos**, todėl mažėja prognozės tikslumas.
- Nors binning modelio rezultatai šiek tiek stabilesni, vidutinė prognozės kokybė yra žemesnė.

### Verslo išvada
- Jei tikslas yra **maksimalus prognozės tikslumas**, prioritetas turėtų būti žali skaitiniai kintamieji.
- Jei svarbiausia **aiški segmentinė interpretacija**, binning vis tiek gali būti pateisinamas.

### Praktinė išvada
- Binning neturėtų būti naudojamas automatiškai.
- Sprendimas taikyti binning turi būti pagrįstas **CV metrikų palyginimu**, o ne vien interpretacijos patogumu.

**Esmė:** binning didina aiškumą, bet dažniausiai mažina prognozinę galią – kompromisą reikia rinktis sąmoningai.


## 11. Dažnos klaidos ir gerosios praktikos (santrauka)

Dažnos klaidos:
- Per daug polinomo laipsnio be CV (overfitting).
- Interaction term be originalių kintamųjų (neteisinga interpretacija).
- Kategorijų one-hot encoding be reference level (tobula multikolinearumo).
- Retos kategorijos paliekamos atskirai (nestabilūs koeficientai, variance).
- Tipų nesuderinamumas (`object` / `bool`) pateikiamas į `statsmodels` (galimi `ValueError`).

Gerosios praktikos:
- Naujas idėjas tikrinti su **cross-validation**.
- Kategorijas paversti į skaitinius su `pd.get_dummies(...).astype(float)`.
- Naudoti `drop_first=True` arba rankiniu būdu pasirinkti reference level.
- Retas kategorijas jungti į „Other“ arba į versliškai prasmingas grupes.
- Vertinti ne tik metriką, bet ir interpretacijos aiškumą.


## 12. Key takeaways

- Feature engineering paverčia žalius duomenis į požymius, kurie **gerina modelio tikslumą** ir **interpretaciją**.
- Skaitiniams kintamiesiems taikomi: polinomai, kombinacijos, interaction terms, binning.
- Kategoriniai kintamieji paruošiami per: binary columns, dummy variables, kategorijų grupavimą.
- Daugiausia vertės dažniausiai duoda domeno supratimas: aiški hipotezė, kas turėtų veikti tikslinį kintamąjį.
